<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #FFFFFF; max-width: 90%; overflow-x: auto; color: #000000;">

<img src="../../resources/swdb_logo.jpg">

<h1 align="center">Mini-Workshop 2: Selection Bias in Tuning Curves</h1>
<h3 align="center">Summer Workshop on the Dynamic Brain</h3>
<h4 align="center">Thursday, August 27th, 2026</h4>
<h4 align="center">Day 4</h4>

---

***Authors:** Nick Steinmetz, Carrie Stine*

---

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

# The Setup: 
An orientation-tuning experiment was run in two conditions **(control and test)**.
For 200 V1 neurons, spike counts were recorded at 12 directions (30-degree
spacing, 10 repeats each). A colleague ran the standard analysis below and
concluded that **the test condition reduces neural responsiveness.**

Run the cells to reproduce the analysis, then work through the exercise to
evaluate whether the conclusion holds up.

</div>

In [ ]:
# Setup (imports and plotting defaults)
import os
os.chdir('/code/mini-workshops/nb2')
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['font.sans-serif']   = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['pdf.fonttype']      = 42
plt.rcParams['ps.fonttype']       = 42
CONTROL, TEST = 0, 1
COND_COLOR    = {'control': 'k', 'test': '0.6'}
COND_LABEL    = {'control': 'Control', 'test': 'Test'}
rng = np.random.default_rng(0)

def load(fname):
    """Load spike count dataset. Returns counts, duration, orientations, condition names."""
    d = np.load(fname, allow_pickle=True)
    return (d['counts'], float(d['duration']), d['orientations'],
            [str(x) for x in d['condition_names']])

def preferred_ori(rate_by_ori):
    """Find each neuron's preferred direction, breaking ties at random.
    Counts are small integers so exact ties are common; np.argmax's first-index
    rule would bias the preferred direction toward 0 deg."""
    return (rate_by_ori + rng.uniform(0, 1e-9, rate_by_ori.shape)).argmax(1)

def roll_to_center(rate_by_ori, pref_idx):
    """Circularly shift each neuron's tuning curve so its preferred direction
    sits at the center column, enabling averaging across neurons."""
    n_ori  = rate_by_ori.shape[1]
    center = n_ori // 2
    return np.array([np.roll(rate_by_ori[i], center - pref_idx[i])
                     for i in range(rate_by_ori.shape[0])])


In [ ]:
# Load the dataset and print its shape
counts, duration, orientations, condition_names = load('data/tuned.npz')
n_neurons, n_ori, n_cond, n_trials = counts.shape
print(f'{n_neurons} neurons x {n_ori} directions x {n_cond} conditions x {n_trials} trials')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Step 1 — Mean firing rate, then align to each neuron's preferred direction

1. Compute each neuron's mean firing rate across trials for each direction and condition.
2. Define each neuron's **preferred direction** as the direction with the highest
   mean response in the **control** condition.
3. Align both conditions to each neuron's preferred direction (shift so the preferred
   direction sits at the center) and average across neurons.

</div>


In [ ]:
rate      = counts / duration                    # spike counts -> firing rate (spikes/s)
mean_rate = rate.mean(axis=3)                    # average over trials -> (neuron, ori, condition)

# Define preferred direction from the CONTROL condition, then align both conditions to it.
pref_idx        = preferred_ori(mean_rate[:, :, CONTROL])
aligned_control = roll_to_center(mean_rate[:, :, CONTROL], pref_idx)
aligned_test    = roll_to_center(mean_rate[:, :, TEST],    pref_idx)

center  = n_ori // 2
rel_ori = (np.arange(n_ori) - center) * (orientations[1] - orientations[0])


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Step 2 — Plot the mean tuning curve in control vs test conditions

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
for cond, curves in [('control', aligned_control), ('test', aligned_test)]:
    m   = curves.mean(0)
    sem = curves.std(0) / np.sqrt(n_neurons)
    ax.errorbar(rel_ori, m, yerr=sem, color=COND_COLOR[cond], lw=2,
                marker='o', ms=4, capsize=2, label=COND_LABEL[cond])
ax.set_xlabel('direction relative to preferred (deg)')
ax.set_ylabel('evoked firing rate (spikes/s)')
ax.set_title('Mean tuning curve (n = %d)' % n_neurons)
ax.legend(frameon=False)
plt.show()

# Test at the preferred direction (center column), neuron by neuron.
p_ctrl = aligned_control[:, center]
p_test = aligned_test[:, center]
w      = wilcoxon(p_ctrl, p_test)
print(f'Firing rate at preferred direction:')
print(f'  Control : {p_ctrl.mean():.2f} +/- {p_ctrl.std()/np.sqrt(n_neurons):.2f} sp/s')
print(f'  Test    : {p_test.mean():.2f} +/- {p_test.std()/np.sqrt(n_neurons):.2f} sp/s')
print(f'  Wilcoxon signed-rank test: p = {w.pvalue:.2e}')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

The analysis shows clear tuning with a large, highly significant drop at the
preferred direction in the test condition (p < 1e-5). A colleague concludes:

> *"These neurons are orientation tuned, and the test manipulation reduces
> responsiveness: at each neuron's preferred orientation, the evoked firing
> rate is markedly higher in control than test."*

**The key conclusion (the reduction) is not supported by this analysis.** Work
through the exercise below to find out why.

</div>


<div style="border-left: 3px solid #07bc0a; padding: 1px; padding-left: 10px; background: #DFF0D8; max-width: 90%; overflow-x: auto; color: #000000;">

### Exercise: Evaluate the tuning curve analysis

<ol>

<li><strong>Predict the outcome if you select from the other condition.</strong> If the
reduction were real, it should not matter which condition defines the preferred
direction. Predict: what would the figure look like if you used the
<em>test</em> condition to define preferred direction instead of control?
<details>
<summary>Hint</summary>

*Whatever bias exists, ask yourself: which condition "earned" the preferred
direction label in each case? Does the direction of the apparent difference
follow the biology or the selection?*

</details>
</li>

<br>

<li><strong>Design a fair test.</strong> How would you separately answer (a) are these
neurons genuinely tuned, and (b) is the condition difference real? What data
would each analysis use to define the preferred direction, and what data
would it use to measure?
<details>
<summary>Hint</summary>

*You have multiple trials per condition. Could you use one subset to decide
which direction to look at, and a completely separate subset to actually
measure? Would the test trials ever need to be used for the selection step? What
principles of cross-validation have we learned earlier in today's workshops that 
we could apply here?*

</details>
</li>

</ol>

</div>


In [ ]:
# 1. Predict the outcome if you select from the other condition.
# If the reduction were real, swapping to select from TEST should not reverse it.



In [ ]:
# 2. Define a fair test (cross-validation): for many random splits of control trials, define the
# preferred direction on one half and read out on the other half (and test).


<div style="border-left: 3px solid #07bc0a; padding: 1px; padding-left: 10px; background: #DFF0D8; max-width: 90%; overflow-x: auto; color: #000000;">

### Exercise: Evaluate the analysis workflow on untuned neurons

The dataset `untuned.npz` has **no orientation tuning**: every spike
count in every condition is drawn from the *same* Poisson rate. Does the same analysis pipeline
still produce a sharp tuning curve and a large condition difference?

<ol>

<li><strong>Repeat the exact same analysis workflow on the untuned data.</strong> Do you still see an effect?

</li>

<br>

<li><strong>Repeat the cross-validation on the untuned data.</strong> Is the result different from what you got with the real data? If so, can you figure out why?
</li>

</ol>

</div>


In [ ]:
# 1. Load the untuned population and repeat the analysis pipeline.


In [ ]:
# 2. Repeat the cross-validation on the untuned population.


<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

## Key takeaways

<details>
<summary><b>reveal after completing the exercises!</b></summary>

Both datasets were generated with control and test conditions identical - there
is no condition difference anywhere. The apparent reduction is entirely a
selection artifact.

- **Never select and measure on the same data (circular analysis / double dipping).**
  Choosing the preferred direction by argmax and then reporting the response *at
  that direction* biases the selected value upward. Any statistic used to select
  must be computed on data independent of the statistic you report.
- **Selection bias can inflate a comparison or manufacture the whole effect.**
  With real tuning it invents a condition difference; with no tuning it
  invents the tuning curve itself.
- **The bias follows whichever condition you select on.** If the reduction were
  real it would not flip when you swap which condition defines the preferred direction.
- **The winner's curse is regression to the mean**, biggest when estimates are
  noisy (few spikes) and when many directions sit near the peak (broad tuning).
- **Cross-validation separates the two questions.** It correctly shows that the
  tuning is real (the peak survives independent data) while the condition
  difference is not (it vanishes with independent selection).

**How to do it right:** define the preferred direction on one set of trials and
quantify tuning / condition differences on independent trials
(cross-validation); or avoid peak-alignment and compare conditions with an
alignment-free model (a fitted tuning function, or the full direction x condition
response in a proper repeated-measures model). Do that here and the condition
difference is zero -- as it must be, since control and test were generated
identically.

</details>

<br>

</div>
